##### Imports and setup

In [1]:
# setup the notebook environment
!pixi install
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import logging
logging.getLogger("pypsa.network.descriptors").setLevel(logging.ERROR)

def ensure_project_root_on_path(marker_dir: str = "modules") -> str:
    """Find the nearest ancestor folder containing `marker_dir`, add it to
    sys.path (front) and return its path. Raises RuntimeError if not found.
    """
    cwd = Path.cwd().resolve()
    # If cwd contains marker_dir, use cwd
    if (cwd / marker_dir).exists():
        p = str(cwd)
        if p not in sys.path:
            sys.path.insert(0, p)
        return p
    # Walk parents
    for parent in cwd.parents:
        if (parent / marker_dir).exists():
            p = str(parent)
            if p not in sys.path:
                sys.path.insert(0, p)
            return p
    raise RuntimeError(f"Could not find '{marker_dir}' directory in {cwd} or any parent")


project_root = ensure_project_root_on_path()
print("Added project root to sys.path:", project_root)

# imports and loading data
from modules.analysis_toolkit.analyzer import ResultsComputer, GeoOptions
from modules.analysis_toolkit.helpers.plotting import TimeSeriesPlot, HistogramPlot, BarChartPlot, WaterfallPlot
import pandas as pd

study_years = [2030, 2040]
rc = {year: ResultsComputer(year=year) for year in study_years}

✔ The default environment has been installed.
Added project root to sys.path: /Users/rca/PycharmProjects/NGV-FBMC


INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Status Quo (SQ) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Integrated Energy Market (IEM) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'Flow-based market coupling (FBMC) - 2030' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, storage_units, stores, sub_networks
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.6)
INFO:pypsa.network.io:Imported network 'SQ (2030) - redispatch' has buses, carriers, generators, global_constraints, lines, links, loads, shapes, sto

## Average DAM consumer prices in GB

In [5]:
pd.concat(
    [
        rc[year].consumer_costs_in_gb.compare_dispatch() \
            .groupby("bus", axis=0).sum() \
            .groupby("scenario", axis=1).sum() \
            .abs() \
            .sum() \
            .div(
                rc[year].withdrawal.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
                    .drop("Line", level="component") \
                    .drop(["DC", "DC_OH"], level="carrier") \
                    .query("bus.str.contains('GB ')") \
                    .groupby("bus", axis=0).sum() \
                    .groupby("scenario", axis=1).sum() \
                    .sum()
            ) \
            .round(2) \
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
).loc[["iem", "iem_fb", "sq"]]

Calling method consumer_costs_in_gb


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_50383/268934157.py:4: FutureWarning:

The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.

/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_50383/268934157.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method withdrawal


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_50383/268934157.py:13: FutureWarning:

The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.

/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_50383/268934157.py:14: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method consumer_costs_in_gb


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_50383/268934157.py:4: FutureWarning:

The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.

/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_50383/268934157.py:5: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method withdrawal


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_50383/268934157.py:13: FutureWarning:

The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.

/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_50383/268934157.py:14: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
iem,27.32,3.06
iem_fb,27.37,3.03
sq,27.06,3.07


# Consumer costs

In [ ]:
# TODO compute the total consumer costs, including all components

## Dispatch consumer costs

In [2]:
pd.concat(
    [
        rc[year].consumer_costs_in_gb.compare_dispatch(bus_carrier=["AC", "AC_OH"], groupby=["bus", "carrier"]) \
            .groupby("scenario", axis=1).sum() \
            .sum()
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method consumer_costs_in_gb


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_50383/1401095795.py:4: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method consumer_costs_in_gb


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_50383/1401095795.py:4: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year,2030,2040
scenario,,
diff: iem-sq,-93.3,2.2
diff: iemfb-iem,-17.6,25.8
iem,-9862.3,-1939.4
iem_fb,-9879.9,-1913.6
sq,-9769.0,-1941.6


## CfD compensation

In [ ]:
# TODO: when Aisling shares the strike prices

## Redispatch costs

In [6]:
pd.concat(
    [
        rc[year].constraint_costs.compare_redispatch()
            .sum() \
            .groupby("scenario").sum() \
            .div(1e6) \
            .round(1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method constraint_costs
Calling method constraint_costs


year,2030,2040
scenario,,
diff: iem-sq,-2538.4,-7717.0
diff: iemfb-iem,49.4,-494.2
iem,807.1,11115.6
iem_fb,856.5,10621.3
sq,3345.5,18832.6


## Compensation to merchant interconnectors (eventual)

In [19]:
pd.concat(
    [
        rc[year].lost_congestion_income() \
            .div(1e6) \
            .round(1) \
            .sum(axis=1)
            .agg({"name": "sum"})
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method: lost_congestion_income
Calling method congestion_income
Calling method congestion_income
Calling method: lost_congestion_income
Calling method congestion_income
Calling method congestion_income


year,2030,2040
name,4.4,12.8


### breakdown per interconnector
In 2030, BritNed, IFA and IFA2 and Eleclink lose congestion income due to flow-based constraints, adding up to 4.4 M€.
In 2040, some interconnectors lose but others gain congestion income, with a net loss of 12.8 M€.

In [18]:
pd.concat(
    [
        rc[year].lost_congestion_income() \
            .div(1e6) \
            .round(1) \
            .sum(axis=1)
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

Calling method: lost_congestion_income
Calling method congestion_income
Calling method congestion_income
Calling method: lost_congestion_income
Calling method congestion_income
Calling method congestion_income


year,2030,2040
name,,
Aminth,0.0,2.3
BritNed,0.9,-1.4
Continental Link,0.0,0.0
Cronos,0.0,-1.6
East-West,0.0,0.0
ElecLink,1.3,1.5
FAB Link,0.0,4.0
Greenlink (Greenwire),0.0,0.0
Gridlink,0.0,-3.4


# Merchant CI

In [13]:
# note: run each of the components first
pd.concat({
    "SQ": captured_sq,
    "IEM": captured_sq + delta_price_coupling + delta_capture_rate,
    "IEM-FB": captured_sq + delta_price_coupling + delta_capture_rate + delta_gb_constraints,
})
# assuming no compensation for IEM-FB

,year,2030,2040
SQ,name,3632.8,3307.5
IEM,name,4558.5,4259.3
IEM-FB,name,4554.1,4246.5


## Captured SQ CI

In [6]:
captured_sq = pd.concat(
    [
        rc[year].congestion_income.sq_dispatch(where=GeoOptions.GB_ONLY, apply_capture_rates=True) \
            .div(1e6) \
            .round(1) \
            .sum(axis=1) \
            .agg({"name": "sum"})
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

captured_sq

Calling method congestion_income
Calling method congestion_income


year,2030,2040
name,3632.8,3307.5


## ∆ price coupling (price and volume effects)

In [7]:
delta_price_coupling = pd.concat(
    [
        rc[year].congestion_income.compare_dispatch(where=GeoOptions.GB_ONLY) \
            .div(1e6) \
            .round(1) \
            .groupby("scenario", axis=1).sum() \
            .sum()
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
).loc["diff: iem-sq"]

delta_price_coupling

Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_65242/4044576906.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_65242/4044576906.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year
2030   -10.5
2040    -8.7
Name: diff: iem-sq, dtype: float64

## ∆ capture rate

In [8]:
delta_capture_rate = pd.concat(
    [
        rc[year].congestion_income.sq_dispatch(where=GeoOptions.GB_ONLY, apply_capture_rates=False) \
            .div(1e6) \
            .round(1) \
            .sum(axis=1) \
            .agg({"name": "sum"}).sub(
                rc[year].congestion_income.sq_dispatch(where=GeoOptions.GB_ONLY, apply_capture_rates=True) \
                    .div(1e6) \
                    .round(1) \
                    .sum(axis=1) \
                    .agg({"name": "sum"})
            )
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

delta_capture_rate

Calling method congestion_income
Calling method congestion_income
Calling method congestion_income
Calling method congestion_income


year,2030,2040
name,936.2,960.5


## ∆ GB constraints (FB reflecting GB grid in dispatch)

In [9]:
delta_gb_constraints = pd.concat(
    [
        rc[year].congestion_income.compare_dispatch(where=GeoOptions.GB_ONLY) \
            .div(1e6) \
            .round(1) \
            .groupby("scenario", axis=1).sum() \
            .sum()
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
).loc["diff: iemfb-iem"]

delta_gb_constraints

Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_65242/3061927899.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



Calling method congestion_income


/var/folders/9b/zpm4mfyd03s_22tcv6xfdv780000gn/T/ipykernel_65242/3061927899.py:6: FutureWarning:

DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.



year
2030    -4.4
2040   -12.8
Name: diff: iemfb-iem, dtype: float64

## ∆ compensation flow limits

In [10]:
delta_compensation = pd.concat(
    [
        rc[year].lost_congestion_income() \
            .div(1e6) \
            .round(1) \
            .sum(axis=1)
            .agg({"name": "sum"})
        for year in study_years
    ],
    keys=study_years,
    names=["year"],
    axis=1
)

delta_compensation

Calling method: lost_congestion_income
Calling method congestion_income
Calling method congestion_income
Calling method: lost_congestion_income
Calling method congestion_income
Calling method congestion_income


year,2030,2040
name,4.4,12.8
